In [ ]:
import torch
import numpy as np
import time
import argparse
import sys
from typing import Dict, List, Optional, Tuple
import psutil
import platform
from transformers import AutoTokenizer, AutoModel
from calflops import calculate_flops_hf
import warnings
warnings.filterwarnings("ignore")

class HardwareBenchmark:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.supported_dtypes = {
            'f32': torch.float32,
            'f16': torch.float16,
            'bf16': torch.bfloat16,
            'int16': torch.int16,
            'int8': torch.int8,
        }
        
        # Quantized types simulation (actual quantization would need specific libraries)
        self.quantized_types = ['q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8']
        
    def get_system_info(self):
        """Get system information"""
        info = {
            'Platform': platform.system(),
            'Processor': platform.processor(),
            'Architecture': platform.machine(),
            'CPU Cores': psutil.cpu_count(logical=False),
            'Logical CPUs': psutil.cpu_count(logical=True),
            'Total RAM': f"{psutil.virtual_memory().total / (1024**3):.2f} GB",
            'Python Version': platform.python_version(),
            'PyTorch Version': torch.__version__,
        }
        
        if torch.cuda.is_available():
            info['GPU'] = torch.cuda.get_device_name(0)
            info['GPU Memory'] = f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
            info['CUDA Version'] = torch.version.cuda
        else:
            info['GPU'] = 'Not available'
            
        return info
    
    def print_system_info(self):
        """Print system information"""
        print("\n" + "="*50)
        print("SYSTEM INFORMATION")
        print("="*50)
        info = self.get_system_info()
        for key, value in info.items():
            print(f"{key:15}: {value}")
        print("="*50)

class MatrixBenchmark(HardwareBenchmark):
    def __init__(self):
        super().__init__()
        self.matrix_sizes = {
            'small': (512, 512),
            'medium': (1024, 1024),
            'large': (2048, 2048),
            'xlarge': (4096, 4096)
        }
    
    def create_matrix(self, size: Tuple[int, int], dtype: torch.dtype, device: torch.device) -> torch.Tensor:
        """Create a random matrix with specified size and dtype"""
        if dtype in [torch.int8, torch.int16]:
            return torch.randint(-128, 127, size, dtype=dtype, device=device)
        else:
            return torch.randn(size, dtype=dtype, device=device)
    
    def benchmark_matrix_operations(self, size_name: str, dtype_name: str, iterations: int = 10) -> Dict:
        """Benchmark matrix operations for a specific size and dtype"""
        size = self.matrix_sizes[size_name]
        dtype = self.supported_dtypes[dtype_name]
        
        # Skip unsupported combinations
        if dtype == torch.bfloat16 and not torch.cuda.is_available():
            return {'error': 'bfloat16 requires CUDA'}
        
        try:
            # Create matrices
            A = self.create_matrix(size, dtype, self.device)
            B = self.create_matrix(size, dtype, self.device)
            
            # Warm up
            for _ in range(3):
                _ = torch.mm(A, B)
            
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Benchmark matrix multiplication
            start_time = time.time()
            for _ in range(iterations):
                C = torch.mm(A, B)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
            end_time = time.time()
            
            # Calculate metrics
            total_time = end_time - start_time
            avg_time = total_time / iterations
            
            # Calculate FLOPs (2 * M * N * K for matrix multiplication)
            flops = 2 * size[0] * size[1] * size[1]
            gflops = (flops * iterations) / (total_time * 1e9)
            
            # Memory usage
            matrix_memory = A.element_size() * A.nelement() * 3  # A, B, C
            memory_gb = matrix_memory / (1024**3)
            
            return {
                'avg_time_ms': avg_time * 1000,
                'gflops': gflops,
                'memory_gb': memory_gb,
                'dtype_size': A.element_size(),
                'total_elements': A.nelement(),
                'device': str(self.device)
            }
            
        except Exception as e:
            return {'error': str(e)}
    
    def run_matrix_benchmark(self):
        """Run comprehensive matrix benchmark"""
        print("\n" + "="*60)
        print("MATRIX OPERATIONS BENCHMARK")
        print("="*60)
        
        results = {}
        
        for size_name in self.matrix_sizes.keys():
            print(f"\nTesting {size_name.upper()} matrices {self.matrix_sizes[size_name]}:")
            print("-" * 50)
            
            for dtype_name in self.supported_dtypes.keys():
                print(f"Testing {dtype_name}...", end=" ")
                
                result = self.benchmark_matrix_operations(size_name, dtype_name)
                results[f"{size_name}_{dtype_name}"] = result
                
                if 'error' in result:
                    print(f"❌ {result['error']}")
                else:
                    print(f"✅ {result['avg_time_ms']:.2f}ms, {result['gflops']:.2f} GFLOPS")
        
        # Print summary table
        self.print_matrix_summary(results)
        return results
    
    def print_matrix_summary(self, results: Dict):
        """Print a formatted summary of matrix benchmark results"""
        print("\n" + "="*80)
        print("MATRIX BENCHMARK SUMMARY")
        print("="*80)
        print(f"{'Size':<8} {'Dtype':<6} {'Time(ms)':<10} {'GFLOPS':<10} {'Memory(GB)':<12} {'Status'}")
        print("-" * 80)
        
        for key, result in results.items():
            size, dtype = key.split('_')
            if 'error' in result:
                print(f"{size:<8} {dtype:<6} {'N/A':<10} {'N/A':<10} {'N/A':<12} ❌ {result['error']}")
            else:
                print(f"{size:<8} {dtype:<6} {result['avg_time_ms']:<10.2f} {result['gflops']:<10.2f} {result['memory_gb']:<12.4f} ✅")

class ModelBenchmark(HardwareBenchmark):
    def __init__(self):
        super().__init__()
        self.test_models = {
            'small': 'distilbert-base-uncased',
            'medium': 'bert-base-uncased', 
            'large': 'microsoft/DialoGPT-medium',
            'xlarge': 'microsoft/DialoGPT-large'
        }
    
    def calculate_model_flops(self, model_name: str) -> Dict:
        """Calculate FLOPs for a model using calflops"""[8]
        try:
            # Using the calflops library mentioned in the search results
            flops, macs, params = calculate_flops_hf(
                model_name_or_path=model_name,
                input_shape=(1, 128),  # batch_size=1, sequence_length=128
                transformer_tokenizer_name=model_name,
                device=self.device
            )
            
            return {
                'flops': flops,
                'macs': macs,
                'params': params,
                'model_name': model_name
            }
        except Exception as e:
            return {'error': str(e)}
    
    def benchmark_inference(self, model_name: str, num_iterations: int = 10) -> Dict:
        """Benchmark inference speed for a model"""
        try:
            # Load model and tokenizer
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModel.from_pretrained(model_name).to(self.device)
            
            # Prepare input
            text = "This is a test sentence for benchmarking inference speed."
            inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            # Warm up
            model.eval()
            with torch.no_grad():
                for _ in range(3):
                    _ = model(**inputs)
            
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Benchmark inference
            start_time = time.time()
            with torch.no_grad():
                for _ in range(num_iterations):
                    outputs = model(**inputs)
                    if torch.cuda.is_available():
                        torch.cuda.synchronize()
            end_time = time.time()
            
            total_time = end_time - start_time
            avg_time = total_time / num_iterations
            throughput = num_iterations / total_time
            
            return {
                'avg_inference_time_ms': avg_time * 1000,
                'throughput_samples_per_sec': throughput,
                'total_time_sec': total_time,
                'model_name': model_name,
                'device': str(self.device)
            }
            
        except Exception as e:
            return {'error': str(e)}
    
    def benchmark_training_step(self, model_name: str, num_iterations: int = 5) -> Dict:
        """Benchmark training step speed"""[3]
        try:
            from transformers import AutoModelForSequenceClassification
            
            # Load model for training
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name, 
                num_labels=2,
                ignore_mismatched_sizes=True
            ).to(self.device)
            
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            # Prepare training data
            texts = ["This is a positive example."] * 4
            labels = [1, 0, 1, 0]
            
            inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=128)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            labels = torch.tensor(labels).to(self.device)
            
            # Setup optimizer
            optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
            
            # Warm up
            model.train()
            for _ in range(2):
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
            
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Benchmark training steps
            start_time = time.time()
            for _ in range(num_iterations):
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
            end_time = time.time()
            
            total_time = end_time - start_time
            avg_time = total_time / num_iterations
            throughput = num_iterations / total_time
            
            return {
                'avg_training_step_time_ms': avg_time * 1000,
                'throughput_steps_per_sec': throughput,
                'total_time_sec': total_time,
                'model_name': model_name,
                'device': str(self.device)
            }
            
        except Exception as e:
            return {'error': str(e)}
    
    def run_model_benchmark(self, download_models: bool = True, test_training: bool = False, test_inference: bool = True):
        """Run model benchmark with user preferences"""
        print("\n" + "="*60)
        print("MODEL BENCHMARK")
        print("="*60)
        
        if not download_models:
            print("Skipping model download - using FLOPS calculation only")
            
        results = {}
        
        for size_name, model_name in self.test_models.items():
            print(f"\nTesting {size_name.upper()} model: {model_name}")
            print("-" * 50)
            
            # Calculate FLOPs (doesn't require full model download)
            print("Calculating FLOPs...", end=" ")
            flops_result = self.calculate_model_flops(model_name)
            results[f"{size_name}_flops"] = flops_result
            
            if 'error' in flops_result:
                print(f"❌ {flops_result['error']}")
            else:
                print(f"✅ {flops_result['flops']:.2e} FLOPs, {flops_result['params']:.2e} params")
            
            if download_models:
                # Test inference
                if test_inference:
                    print("Testing inference...", end=" ")
                    inf_result = self.benchmark_inference(model_name)
                    results[f"{size_name}_inference"] = inf_result
                    
                    if 'error' in inf_result:
                        print(f"❌ {inf_result['error']}")
                    else:
                        print(f"✅ {inf_result['avg_inference_time_ms']:.2f}ms/sample")
                
                # Test training
                if test_training:
                    print("Testing training step...", end=" ")
                    train_result = self.benchmark_training_step(model_name)
                    results[f"{size_name}_training"] = train_result
                    
                    if 'error' in train_result:
                        print(f"❌ {train_result['error']}")
                    else:
                        print(f"✅ {train_result['avg_training_step_time_ms']:.2f}ms/step")
        
        self.print_model_summary(results)
        return results
    
    def print_model_summary(self, results: Dict):
        """Print formatted model benchmark summary"""
        print("\n" + "="*80)
        print("MODEL BENCHMARK SUMMARY")
        print("="*80)
        print(f"{'Model':<10} {'Test':<12} {'FLOPs':<12} {'Params':<12} {'Time(ms)':<12} {'Status'}")
        print("-" * 80)
        
        for key, result in results.items():
            parts = key.split('_')
            model_size = parts[0]
            test_type = '_'.join(parts[1:])
            
            if 'error' in result:
                print(f"{model_size:<10} {test_type:<12} {'N/A':<12} {'N/A':<12} {'N/A':<12} ❌ {result['error']}")
            else:
                if test_type == 'flops':
                    flops = f"{result['flops']:.1e}" if 'flops' in result else 'N/A'
                    params = f"{result['params']:.1e}" if 'params' in result else 'N/A'
                    print(f"{model_size:<10} {test_type:<12} {flops:<12} {params:<12} {'N/A':<12} ✅")
                else:
                    time_key = 'avg_inference_time_ms' if 'inference' in result else 'avg_training_step_time_ms'
                    time_val = f"{result[time_key]:.2f}" if time_key in result else 'N/A'
                    print(f"{model_size:<10} {test_type:<12} {'N/A':<12} {'N/A':<12} {time_val:<12} ✅")

def main():
    parser = argparse.ArgumentParser(description='Hardware and Model Benchmark Tool')
    parser.add_argument('--test-type', choices=['matrix', 'model', 'both'], default='both',
                       help='Type of benchmark to run')
    parser.add_argument('--download-models', action='store_true',
                       help='Download and test actual models (requires internet)')
    parser.add_argument('--test-training', action='store_true',
                       help='Test training performance (requires model download)')
    parser.add_argument('--test-inference', action='store_true', default=True,
                       help='Test inference performance (requires model download)')
    
    args = parser.parse_args()
    
    # Interactive mode if no arguments provided
    if len(sys.argv) == 1:
        print("Welcome to Hardware & Model Benchmark Tool!")
        print("=" * 50)
        
        # Ask for test type
        print("\nWhich test would you like to run?")
        print("1. Matrix operations benchmark")
        print("2. Model benchmark")
        print("3. Both")
        
        while True:
            choice = input("Enter your choice (1-3): ").strip()
            if choice == '1':
                args.test_type = 'matrix'
                break
            elif choice == '2':
                args.test_type = 'model'
                break
            elif choice == '3':
                args.test_type = 'both'
                break
            else:
                print("Invalid choice. Please enter 1, 2, or 3.")
        
        # Ask about model download
        if args.test_type in ['model', 'both']:
            while True:
                download = input("\nDownload models for real testing? (y/n): ").strip().lower()
                if download in ['y', 'yes']:
                    args.download_models = True
                    break
                elif download in ['n', 'no']:
                    args.download_models = False
                    break
                else:
                    print("Please enter 'y' or 'n'.")
            
            # Ask about training/inference
            if args.download_models:
                while True:
                    test_inf = input("Test inference performance? (y/n): ").strip().lower()
                    if test_inf in ['y', 'yes']:
                        args.test_inference = True
                        break
                    elif test_inf in ['n', 'no']:
                        args.test_inference = False
                        break
                    else:
                        print("Please enter 'y' or 'n'.")
                
                while True:
                    test_train = input("Test training performance? (y/n): ").strip().lower()
                    if test_train in ['y', 'yes']:
                        args.test_training = True
                        break
                    elif test_train in ['n', 'no']:
                        args.test_training = False
                        break
                    else:
                        print("Please enter 'y' or 'n'.")
    
    # Initialize benchmarks
    matrix_bench = MatrixBenchmark()
    model_bench = ModelBenchmark()
    
    # Print system info
    matrix_bench.print_system_info()
    
    # Run benchmarks
    if args.test_type in ['matrix', 'both']:
        matrix_bench.run_matrix_benchmark()
    
    if args.test_type in ['model', 'both']:
        model_bench.run_model_benchmark(
            download_models=args.download_models,
            test_training=args.test_training,
            test_inference=args.test_inference
        )
    
    print("\n" + "="*50)
    print("BENCHMARK COMPLETED!")
    print("="*50)

if __name__ == "__main__":
    main()


In [22]:

class HardwareBenchmark:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.supported_dtypes = {
            'f32': torch.float32,
            'f16': torch.float16,
            'bf16': torch.bfloat16,
            'int16': torch.int16,
            'int8': torch.int8,
        }
        
        # Quantized types simulation (actual quantization would need specific libraries)
        self.quantized_types = ['q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8']
        
    def get_system_info(self):
        """Get system information"""
        info = {
            'Platform': platform.system(),
            'Processor': platform.processor(),
            'Architecture': platform.machine(),
            'CPU Cores': psutil.cpu_count(logical=False),
            'Logical CPUs': psutil.cpu_count(logical=True),
            'Total RAM': f"{psutil.virtual_memory().total / (1024**3):.2f} GB",
            'Python Version': platform.python_version(),
            'PyTorch Version': torch.__version__,
        }
        
        if torch.cuda.is_available():
            info['GPU'] = torch.cuda.get_device_name(0)
            info['GPU Memory'] = f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
            info['CUDA Version'] = torch.version.cuda
        else:
            info['GPU'] = 'Not available'
            
        return info
    
    def print_system_info(self):
        """Print system information"""
        print("\n" + "="*50)
        print("SYSTEM INFORMATION")
        print("="*50)
        info = self.get_system_info()
        for key, value in info.items():
            print(f"{key:15}: {value}")
        print("="*50)

class MatrixBenchmark(HardwareBenchmark):
    def __init__(self):
        super().__init__()
        self.matrix_sizes = {
            'small': (512, 512),
            'medium': (1024, 1024),
            'large': (2048, 2048),
            'xlarge': (4096, 4096),
            'bigger':(10000,10000)
        }
    
    def create_matrix(self, size: Tuple[int, int], dtype: torch.dtype, device: torch.device) -> torch.Tensor:
        """Create a random matrix with specified size and dtype"""
        if dtype in [torch.int8, torch.int16]:
            return torch.randint(-128, 127, size, dtype=dtype, device=device)
        else:
            return torch.randn(size, dtype=dtype, device=device)
    
    def benchmark_matrix_operations(self, size_name: str, dtype_name: str, iterations: int = 10) -> Dict:
        """Benchmark matrix operations for a specific size and dtype"""
        size = self.matrix_sizes[size_name]
        dtype = self.supported_dtypes[dtype_name]
        
        # Skip unsupported combinations
        if dtype == torch.bfloat16 and not torch.cuda.is_available():
            return {'error': 'bfloat16 requires CUDA'}
        
        try:
            # Create matrices
            A = self.create_matrix(size, dtype, self.device)
            B = self.create_matrix(size, dtype, self.device)
            
            # Warm up
            for _ in range(3):
                _ = torch.mm(A, B)
            
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Benchmark matrix multiplication
            start_time = time.time()
            for _ in range(iterations):
                C = torch.mm(A, B)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
            end_time = time.time()
            
            # Calculate metrics
            total_time = end_time - start_time
            avg_time = total_time / iterations
            
            # Calculate FLOPs (2 * M * N * K for matrix multiplication)
            flops = 2 * size[0] * size[1] * size[1]
            gflops = (flops * iterations) / (total_time * 1e9)
            
            # Memory usage
            matrix_memory = A.element_size() * A.nelement() * 3  # A, B, C
            memory_gb = matrix_memory / (1024**3)
            
            return {
                'avg_time_ms': avg_time * 1000,
                'gflops': gflops,
                'memory_gb': memory_gb,
                'dtype_size': A.element_size(),
                'total_elements': A.nelement(),
                'device': str(self.device)
            }
            
        except Exception as e:
            return {'error': str(e)}
    
    def run_matrix_benchmark(self):
        """Run comprehensive matrix benchmark"""
        print("\n" + "="*60)
        print("MATRIX OPERATIONS BENCHMARK")
        print("="*60)
        
        results = {}
        
        for size_name in self.matrix_sizes.keys():
            print(f"\nTesting {size_name.upper()} matrices {self.matrix_sizes[size_name]}:")
            print("-" * 50)
            
            for dtype_name in self.supported_dtypes.keys():
                print(f"Testing {dtype_name}...", end=" ")
                
                result = self.benchmark_matrix_operations(size_name, dtype_name)
                results[f"{size_name}_{dtype_name}"] = result
                
                if 'error' in result:
                    print(f"❌ {result['error']}")
                else:
                    print(f"✅ {result['avg_time_ms']:.2f}ms, {result['gflops']:.2f} GFLOPS")
        
        # Print summary table
        self.print_matrix_summary(results)
        return results
    
    def print_matrix_summary(self, results: Dict):
        """Print a formatted summary of matrix benchmark results"""
        print("\n" + "="*80)
        print("MATRIX BENCHMARK SUMMARY")
        print("="*80)
        print(f"{'Size':<8} {'Dtype':<6} {'Time(ms)':<10} {'GFLOPS':<10} {'Memory(GB)':<12} {'Status'}")
        print("-" * 80)
        
        for key, result in results.items():
            size, dtype = key.split('_')
            if 'error' in result:
                print(f"{size:<8} {dtype:<6} {'N/A':<10} {'N/A':<10} {'N/A':<12} ❌ {result['error']}")
            else:
                print(f"{size:<8} {dtype:<6} {result['avg_time_ms']:<10.2f} {result['gflops']:<10.2f} {result['memory_gb']:<12.4f} ✅")

In [23]:
hw_test = HardwareBenchmark()

mat = MatrixBenchmark()


In [24]:
# mat.create_matrix()
# mat.benchmark_matrix_operations()
res = mat.run_matrix_benchmark()
mat.print_matrix_summary(res)


MATRIX OPERATIONS BENCHMARK

Testing SMALL matrices (512, 512):
--------------------------------------------------
Testing f32... ✅ 0.08ms, 3540.57 GFLOPS
Testing f16... ✅ 0.07ms, 3890.46 GFLOPS
Testing bf16... ✅ 0.12ms, 2202.90 GFLOPS
Testing int16... ❌ "addmm_cuda" not implemented for 'Short'
Testing int8... ❌ "addmm_cuda" not implemented for 'Char'

Testing MEDIUM matrices (1024, 1024):
--------------------------------------------------
Testing f32... ✅ 0.43ms, 5049.44 GFLOPS
Testing f16... ✅ 0.35ms, 6073.22 GFLOPS
Testing bf16... ✅ 0.52ms, 4160.95 GFLOPS
Testing int16... ❌ "addmm_cuda" not implemented for 'Short'
Testing int8... ❌ "addmm_cuda" not implemented for 'Char'

Testing LARGE matrices (2048, 2048):
--------------------------------------------------
Testing f32... ✅ 2.56ms, 6715.84 GFLOPS
Testing f16... ✅ 2.48ms, 6933.75 GFLOPS
Testing bf16... ✅ 3.24ms, 5306.70 GFLOPS
Testing int16... ❌ "addmm_cuda" not implemented for 'Short'
Testing int8... ❌ "addmm_cuda" not implemented

In [31]:
class ModelBenchmark(HardwareBenchmark):
    def __init__(self):
        super().__init__()
        self.test_models = {
            'small': 'distilbert-base-uncased',
            'medium': 'bert-base-uncased', 
            'large': 'microsoft/DialoGPT-medium',
            'xlarge': 'microsoft/DialoGPT-large'
        }
    
    def calculate_model_flops(self, model_name: str) -> Dict:
        """Calculate FLOPs for a model using calflops"""[8]
        try:
            # Using the calflops library mentioned in the search results
            flops, macs, params = calculate_flops_hf(
                model_name_or_path=model_name,
                input_shape=(1, 128),  # batch_size=1, sequence_length=128
                transformer_tokenizer_name=model_name,
                device=self.device
            )
            
            return {
                'flops': flops,
                'macs': macs,
                'params': params,
                'model_name': model_name
            }
        except Exception as e:
            return {'error': str(e)}
    
    def benchmark_inference(self, model_name: str, num_iterations: int = 10) -> Dict:
        """Benchmark inference speed for a model"""
        try:
            # Load model and tokenizer
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModel.from_pretrained(model_name).to(self.device)
            
            # Prepare input
            text = "This is a test sentence for benchmarking inference speed."
            inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            # Warm up
            model.eval()
            with torch.no_grad():
                for _ in range(3):
                    _ = model(**inputs)
            
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Benchmark inference
            start_time = time.time()
            with torch.no_grad():
                for _ in range(num_iterations):
                    outputs = model(**inputs)
                    if torch.cuda.is_available():
                        torch.cuda.synchronize()
            end_time = time.time()
            
            total_time = end_time - start_time
            avg_time = total_time / num_iterations
            throughput = num_iterations / total_time
            
            return {
                'avg_inference_time_ms': avg_time * 1000,
                'throughput_samples_per_sec': throughput,
                'total_time_sec': total_time,
                'model_name': model_name,
                'device': str(self.device)
            }
            
        except Exception as e:
            return {'error': str(e)}
    
    def benchmark_training_step(self, model_name: str, num_iterations: int = 5) -> Dict:
        """Benchmark training step speed"""[3]
        try:
            from transformers import AutoModelForSequenceClassification
            
            # Load model for training
            model = AutoModelForSequenceClassification.from_pretrained(
                model_name, 
                num_labels=2,
                ignore_mismatched_sizes=True
            ).to(self.device)
            
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            # Prepare training data
            texts = ["This is a positive example."] * 4
            labels = [1, 0, 1, 0]
            
            inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=128)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            labels = torch.tensor(labels).to(self.device)
            
            # Setup optimizer
            optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
            
            # Warm up
            model.train()
            for _ in range(2):
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
            
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Benchmark training steps
            start_time = time.time()
            for _ in range(num_iterations):
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
            end_time = time.time()
            
            total_time = end_time - start_time
            avg_time = total_time / num_iterations
            throughput = num_iterations / total_time
            
            return {
                'avg_training_step_time_ms': avg_time * 1000,
                'throughput_steps_per_sec': throughput,
                'total_time_sec': total_time,
                'model_name': model_name,
                'device': str(self.device)
            }
            
        except Exception as e:
            return {'error': str(e)}
    
    def run_model_benchmark(self, download_models: bool = True, test_training: bool = False, test_inference: bool = True):
        """Run model benchmark with user preferences"""
        print("\n" + "="*60)
        print("MODEL BENCHMARK")
        print("="*60)
        
        if not download_models:
            print("Skipping model download - using FLOPS calculation only")
            
        results = {}
        
        for size_name, model_name in self.test_models.items():
            print(f"\nTesting {size_name.upper()} model: {model_name}")
            print("-" * 50)
            
            # Calculate FLOPs (doesn't require full model download)
            print("Calculating FLOPs...", end=" ")
            flops_result = self.calculate_model_flops(model_name)
            results[f"{size_name}_flops"] = flops_result
            
            if 'error' in flops_result:
                print(f"❌ {flops_result['error']}")
            else:
                print(f"✅ {flops_result['flops']:.2e} FLOPs, {flops_result['params']:.2e} params")
            
            if download_models:
                # Test inference
                if test_inference:
                    print("Testing inference...", end=" ")
                    inf_result = self.benchmark_inference(model_name)
                    results[f"{size_name}_inference"] = inf_result
                    
                    if 'error' in inf_result:
                        print(f"❌ {inf_result['error']}")
                    else:
                        print(f"✅ {inf_result['avg_inference_time_ms']:.2f}ms/sample")
                
                # Test training
                if test_training:
                    print("Testing training step...", end=" ")
                    train_result = self.benchmark_training_step(model_name)
                    results[f"{size_name}_training"] = train_result
                    
                    if 'error' in train_result:
                        print(f"❌ {train_result['error']}")
                    else:
                        print(f"✅ {train_result['avg_training_step_time_ms']:.2f}ms/step")
        
        self.print_model_summary(results)
        return results
    
    def print_model_summary(self, results: Dict):
        """Print formatted model benchmark summary"""
        print("\n" + "="*80)
        print("MODEL BENCHMARK SUMMARY")
        print("="*80)
        print(f"{'Model':<10} {'Test':<12} {'FLOPs':<12} {'Params':<12} {'Time(ms)':<12} {'Status'}")
        print("-" * 80)
        
        for key, result in results.items():
            parts = key.split('_')
            model_size = parts[0]
            test_type = '_'.join(parts[1:])
            
            if 'error' in result:
                print(f"{model_size:<10} {test_type:<12} {'N/A':<12} {'N/A':<12} {'N/A':<12} ❌ {result['error']}")
            else:
                if test_type == 'flops':
                    flops = f"{result['flops']:.1e}" if 'flops' in result else 'N/A'
                    params = f"{result['params']:.1e}" if 'params' in result else 'N/A'
                    print(f"{model_size:<10} {test_type:<12} {flops:<12} {params:<12} {'N/A':<12} ✅")
                else:
                    time_key = 'avg_inference_time_ms' if 'inference' in result else 'avg_training_step_time_ms'
                    time_val = f"{result[time_key]:.2f}" if time_key in result else 'N/A'
                    print(f"{model_size:<10} {test_type:<12} {'N/A':<12} {'N/A':<12} {time_val:<12} ✅")

In [32]:
model_test = ModelBenchmark()
res = model_test.run_model_benchmark()
model_test.print_model_summary(res)


MODEL BENCHMARK

Testing SMALL model: distilbert-base-uncased
--------------------------------------------------
Calculating FLOPs... ❌ calculate_flops_hf() got an unexpected keyword argument 'model_name_or_path'
Testing inference... ✅ 3.52ms/sample

Testing MEDIUM model: bert-base-uncased
--------------------------------------------------
Calculating FLOPs... ❌ calculate_flops_hf() got an unexpected keyword argument 'model_name_or_path'
Testing inference... ✅ 6.81ms/sample

Testing LARGE model: microsoft/DialoGPT-medium
--------------------------------------------------
Calculating FLOPs... ❌ calculate_flops_hf() got an unexpected keyword argument 'model_name_or_path'
Testing inference... ❌ Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.

Testing XLARGE model: microsoft/DialoGPT-large
---------